# Phase 5 — Gesture Classification (Transformer Encoder)

**Model:** Transformer encoder (4 layers, 4 heads, ~3.8M params)

**Input:** Sliding windows of 15 frames (3s at 5fps) × 132-dim flattened pose keypoints

**5 Gesture Classes:**
- 0 = Illustrator (pointing, showing size — accompanying speech)
- 1 = Emblem (thumbs up, open palm — culturally defined)
- 2 = Beat (rhythmic hand emphasis)
- 3 = Adaptor (self-touching face/hair — nervousness)
- 4 = Rest (hands at sides, clasped, on podium)

**Strategy:** Extract poses from 11 training videos → heuristic auto-label → train Transformer

**Target:** F1 ≥ 0.65

**Setup:** Runtime → Change runtime type → **T4 GPU**

## Cell 1: Install Dependencies & GPU Check

In [ ]:
!pip install -q torch torchvision
!pip install -q mediapipe opencv-python-headless
!pip install -q scikit-learn tqdm

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime -> Change runtime type -> T4 GPU")

## Cell 2: Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/voice_pipeline_models/gesture_transformer"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models will be saved to: {SAVE_DIR}")

# Clone the repo
REPO_DIR = "/content/voice-analysis-pipeline"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/anvay-cpu/voice-analysis-pipeline.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
print(f"Repo at: {REPO_DIR}")

## Cell 3: Download Training Videos

Downloads all 11 training MP4 videos using yt-dlp.
This takes ~5-10 minutes depending on connection speed.

In [ ]:
!pip install -q yt-dlp

VIDEO_DIR = "/content/videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

# All 11 training video URLs
VIDEO_URLS = [
    "https://www.youtube.com/watch?v=YNJfwqKybFY",   # Be Thou Clean - Martin Goury
    "https://www.youtube.com/watch?v=z_pk4eBDaLA",   # Hard Lessons - Stan Druckenmiller
    "https://www.youtube.com/watch?v=M-ZH3psUbfU",   # Elon Musk - Lex Fridman
    "https://www.youtube.com/watch?v=MttW2lFnhKw",   # Tally Feingold - TEDxLFHS
    "https://www.youtube.com/watch?v=ThT0OheCE5M",   # Video 8 (user-provided)
    "https://www.youtube.com/watch?v=9TRSRVmtqMw",   # Video 9 (user-provided)
    "https://www.youtube.com/watch?v=gUV5DJb6KGs",   # Video 10 (user-provided)
    "https://www.youtube.com/watch?v=lKsvLGdoIH8",   # Hot Shot Rule - Kat Cole TED
    "https://www.youtube.com/watch?v=cZJhSpn-Rn4",   # Trusting Our Father - David Homer
    "https://www.youtube.com/watch?v=KW0kDxU7LEg",   # Ancestral Intelligence - Nanjira Sambuli TED
    "https://www.youtube.com/watch?v=ePXyxDmkhNo",   # Power of Body Language - Luca Molina TED
]

existing = [f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')]
if len(existing) >= len(VIDEO_URLS):
    print(f"Videos already downloaded: {len(existing)} MP4 files")
else:
    print(f"Downloading {len(VIDEO_URLS)} videos...")
    for i, url in enumerate(VIDEO_URLS):
        print(f"\n[{i+1}/{len(VIDEO_URLS)}] {url}")
        !yt-dlp -f "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best" \
            --merge-output-format mp4 \
            -o "{VIDEO_DIR}/%(title)s.%(ext)s" \
            "{url}" 2>&1 | tail -3

videos = sorted([f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')])
print(f"\nTotal videos: {len(videos)}")
for v in videos:
    size_mb = os.path.getsize(os.path.join(VIDEO_DIR, v)) / 1e6
    print(f"  {v} ({size_mb:.0f} MB)")

## Cell 4: Download MediaPipe Pose Landmarker Model

In [ ]:
MODEL_PATH = "/content/pose_landmarker_lite.task"

if not os.path.exists(MODEL_PATH):
    !wget -q --show-progress -O {MODEL_PATH} \
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
    print(f"Downloaded: {os.path.getsize(MODEL_PATH) / 1e6:.1f} MB")
else:
    print("Pose model already downloaded.")

## Cell 5: Extract Frames & Pose Keypoints from All Videos

Extracts frames at 5fps and runs MediaPipe pose estimation.
This takes ~20-40 minutes for all 11 videos.

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm
import time

TARGET_FPS = 5
NUM_KEYPOINTS = 33
KEYPOINT_DIM = 4  # x, y, z, visibility

POSE_CACHE_DIR = "/content/pose_cache"
os.makedirs(POSE_CACHE_DIR, exist_ok=True)


def extract_poses_from_video(video_path, model_path, target_fps=5):
    """Extract pose keypoints from video at target FPS.
    
    Returns:
        dict mapping frame_idx -> (33, 4) keypoints or None
    """
    cap = cv2.VideoCapture(video_path)
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_interval = max(1, round(src_fps / target_fps))
    
    # Setup MediaPipe
    base_options = mp.tasks.BaseOptions(model_asset_path=model_path)
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
    )
    
    poses = {}
    out_idx = 0
    frame_num = 0
    
    expected_frames = total_frames // frame_interval
    pbar = tqdm(total=expected_frames, desc="Extracting poses")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_num % frame_interval == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            
            landmarker = mp.tasks.vision.PoseLandmarker.create_from_options(options)
            try:
                result = landmarker.detect(mp_image)
            finally:
                landmarker.close()
            
            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]
                kps = np.zeros((NUM_KEYPOINTS, KEYPOINT_DIM), dtype=np.float32)
                for i, lm in enumerate(landmarks):
                    if i >= NUM_KEYPOINTS:
                        break
                    kps[i] = [lm.x, lm.y, lm.z, lm.visibility or 0.0]
                poses[out_idx] = kps
            else:
                poses[out_idx] = None
            
            out_idx += 1
            pbar.update(1)
        
        frame_num += 1
    
    cap.release()
    pbar.close()
    
    detected = sum(1 for v in poses.values() if v is not None)
    print(f"  Frames: {out_idx}, Detected: {detected} ({100*detected/max(out_idx,1):.1f}%)")
    
    return poses


# Process all videos
all_video_poses = {}  # video_name -> poses dict
videos = sorted([f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')])

for i, vname in enumerate(videos):
    cache_path = os.path.join(POSE_CACHE_DIR, vname.replace('.mp4', '.npz'))
    
    if os.path.exists(cache_path):
        print(f"[{i+1}/{len(videos)}] Loading cached: {vname}")
        data = np.load(cache_path, allow_pickle=True)
        poses = {}
        for key in data.files:
            if key == 'none_indices':
                continue
            idx = int(key.replace('frame_', ''))
            poses[idx] = data[key]
        for idx in data.get('none_indices', []):
            poses[int(idx)] = None
        all_video_poses[vname] = poses
    else:
        print(f"\n[{i+1}/{len(videos)}] Processing: {vname}")
        vpath = os.path.join(VIDEO_DIR, vname)
        start = time.time()
        poses = extract_poses_from_video(vpath, MODEL_PATH, TARGET_FPS)
        elapsed = time.time() - start
        print(f"  Time: {elapsed:.0f}s")
        
        # Cache
        arrays = {}
        none_indices = []
        for idx, kp in poses.items():
            if kp is not None:
                arrays[f'frame_{idx:05d}'] = kp
            else:
                none_indices.append(idx)
        np.savez_compressed(cache_path, none_indices=np.array(none_indices), **arrays)
        all_video_poses[vname] = poses

# Summary
total_frames = sum(len(p) for p in all_video_poses.values())
total_detected = sum(sum(1 for v in p.values() if v is not None) for p in all_video_poses.values())
print(f"\n{'='*60}")
print(f"Total: {total_frames} frames, {total_detected} with pose ({100*total_detected/max(total_frames,1):.1f}%)")
print(f"Videos: {len(all_video_poses)}")

## Cell 6: Create Sliding Windows & Apply Heuristic Labels

Creates 3-second sliding windows (15 frames at 5fps) with ~1.6s hop,
then applies rule-based heuristics to auto-label gesture classes.

In [ ]:
WINDOW_FRAMES = 15   # 3 seconds at 5fps
HOP_FRAMES = 8       # ~1.6 second hop
MIN_VALID_RATIO = 0.8

GESTURE_CLASSES = ["Illustrator", "Emblem", "Beat", "Adaptor", "Rest"]

# Key body landmark indices
L_WRIST, R_WRIST = 15, 16
L_SHOULDER, R_SHOULDER = 11, 12
NOSE = 0
L_EAR, R_EAR = 7, 8
L_HIP, R_HIP = 23, 24
L_ELBOW, R_ELBOW = 13, 14


def heuristic_label(kps_window):
    """Auto-label a (15, 33, 4) keypoint window.
    
    Returns: (class_idx, confidence)
    """
    # Wrist trajectories (x, y)
    l_wrist = kps_window[:, L_WRIST, :2]
    r_wrist = kps_window[:, R_WRIST, :2]
    
    # Reference points
    shoulder_mid = (kps_window[:, L_SHOULDER, :2] + kps_window[:, R_SHOULDER, :2]) / 2
    hip_mid = (kps_window[:, L_HIP, :2] + kps_window[:, R_HIP, :2]) / 2
    nose = kps_window[:, NOSE, :2]
    
    # Total wrist movement (sum of frame-to-frame deltas)
    l_delta = np.sqrt(np.sum(np.diff(l_wrist, axis=0) ** 2, axis=1))
    r_delta = np.sqrt(np.sum(np.diff(r_wrist, axis=0) ** 2, axis=1))
    total_movement = np.sum(l_delta) + np.sum(r_delta)
    
    # Hands near face (adaptor indicator)
    l_face_dist = np.mean(np.sqrt(np.sum((l_wrist - nose) ** 2, axis=1)))
    r_face_dist = np.mean(np.sqrt(np.sum((r_wrist - nose) ** 2, axis=1)))
    min_face_dist = min(l_face_dist, r_face_dist)
    
    # Hands below waist
    l_below = np.mean(l_wrist[:, 1] > hip_mid[:, 1])
    r_below = np.mean(r_wrist[:, 1] > hip_mid[:, 1])
    
    # Hands above shoulders (emblem/illustrator indicator)
    l_above = np.mean(l_wrist[:, 1] < shoulder_mid[:, 1])
    r_above = np.mean(r_wrist[:, 1] < shoulder_mid[:, 1])
    
    # Beat detection: autocorrelation of movement signal
    combined = l_delta + r_delta
    beat_score = 0.0
    if len(combined) > 3 and np.std(combined) > 1e-6:
        normed = (combined - np.mean(combined)) / (np.std(combined) + 1e-8)
        autocorr = np.correlate(normed, normed, mode='full')
        autocorr = autocorr[len(autocorr) // 2:]
        if len(autocorr) > 3:
            autocorr = autocorr / (autocorr[0] + 1e-8)
            beat_score = np.max(autocorr[2:min(7, len(autocorr))])
    
    # Movement symmetry (both hands moving similarly → emblem/beat)
    if np.sum(l_delta) > 0.01 and np.sum(r_delta) > 0.01:
        min_len = min(len(l_delta), len(r_delta))
        if np.std(l_delta[:min_len]) > 0 and np.std(r_delta[:min_len]) > 0:
            symmetry = np.corrcoef(l_delta[:min_len], r_delta[:min_len])[0, 1]
        else:
            symmetry = 0.0
    else:
        symmetry = 0.0
    
    # --- Classification rules ---
    
    # ADAPTOR: hands near face with some movement
    if min_face_dist < 0.08 and total_movement > 0.05:
        return 3, 0.75  # Adaptor
    
    # REST: very low movement OR hands consistently below waist
    if total_movement < 0.12:
        return 4, 0.85  # Rest
    
    if total_movement < 0.20 and (l_below > 0.7 and r_below > 0.7):
        return 4, 0.75  # Rest
    
    # BEAT: rhythmic + symmetric movement
    if beat_score > 0.35 and symmetry > 0.3 and total_movement > 0.15:
        return 2, 0.65  # Beat
    
    # EMBLEM: high symmetry + hands above shoulders
    if symmetry > 0.5 and (l_above > 0.3 or r_above > 0.3):
        return 1, 0.55  # Emblem
    
    # ILLUSTRATOR: active hand movement (default for moving hands)
    if total_movement > 0.15:
        return 0, 0.60  # Illustrator
    
    # Default: Rest
    return 4, 0.50


def create_labeled_windows(poses_dict, window_frames=15, hop_frames=8):
    """Create sliding windows and auto-label them."""
    indices = sorted(poses_dict.keys())
    if not indices:
        return [], []
    
    max_idx = indices[-1]
    windows = []
    labels = []
    confidences = []
    
    for start in range(indices[0], max_idx - window_frames + 2, hop_frames):
        end = start + window_frames
        window_kps = []
        valid = 0
        
        for i in range(start, end):
            kp = poses_dict.get(i)
            if kp is not None:
                window_kps.append(kp)  # (33, 4)
                valid += 1
            else:
                window_kps.append(np.zeros((NUM_KEYPOINTS, KEYPOINT_DIM), dtype=np.float32))
        
        if valid >= MIN_VALID_RATIO * window_frames:
            kps_3d = np.array(window_kps)  # (15, 33, 4)
            label, conf = heuristic_label(kps_3d)
            windows.append(kps_3d.reshape(window_frames, -1))  # (15, 132)
            labels.append(label)
            confidences.append(conf)
    
    return windows, labels, confidences


# Process all videos
all_windows = []
all_labels = []
all_confs = []

for vname, poses in all_video_poses.items():
    wins, labs, confs = create_labeled_windows(poses, WINDOW_FRAMES, HOP_FRAMES)
    all_windows.extend(wins)
    all_labels.extend(labs)
    all_confs.extend(confs)
    print(f"{vname}: {len(wins)} windows")

all_windows = np.array(all_windows, dtype=np.float32)  # (N, 15, 132)
all_labels = np.array(all_labels, dtype=np.int64)       # (N,)
all_confs = np.array(all_confs, dtype=np.float32)       # (N,)

print(f"\n{'='*60}")
print(f"Total windows: {len(all_windows)}")
print(f"Shape: {all_windows.shape}")
print(f"\nClass distribution:")
from collections import Counter
label_counts = Counter(all_labels.tolist())
for i, cls in enumerate(GESTURE_CLASSES):
    count = label_counts.get(i, 0)
    print(f"  {cls:15s}: {count:>5} ({100*count/len(all_labels):.1f}%)")
print(f"\nMean heuristic confidence: {np.mean(all_confs):.3f}")

## Cell 7: Data Augmentation & Train/Val Split

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GestureDataset(Dataset):
    """Dataset of pose keypoint windows with gesture labels."""
    
    def __init__(self, windows, labels, augment=False):
        self.windows = windows  # (N, 15, 132)
        self.labels = labels    # (N,)
        self.augment = augment
    
    def __len__(self):
        return len(self.windows)
    
    def __getitem__(self, idx):
        x = self.windows[idx].copy()  # (15, 132)
        y = self.labels[idx]
        
        if self.augment:
            x = self._augment(x)
        
        return torch.tensor(x, dtype=torch.float32), y
    
    def _augment(self, x):
        """Apply data augmentation."""
        # 1. Joint noise (small random perturbation)
        if np.random.random() < 0.5:
            noise = np.random.normal(0, 0.005, x.shape).astype(np.float32)
            x = x + noise
        
        # 2. Temporal jitter (shift window by ±1-2 frames via rolling)
        if np.random.random() < 0.3:
            shift = np.random.randint(-2, 3)
            x = np.roll(x, shift, axis=0)
        
        # 3. Horizontal flip (swap left/right keypoints)
        if np.random.random() < 0.5:
            x = x.reshape(WINDOW_FRAMES, NUM_KEYPOINTS, KEYPOINT_DIM)
            # Flip x-coordinates
            x[:, :, 0] = 1.0 - x[:, :, 0]
            # Swap left-right keypoint pairs
            lr_pairs = [
                (1, 4), (2, 5), (3, 6),    # eyes
                (7, 8),                      # ears
                (9, 10),                     # mouth
                (11, 12), (13, 14),          # shoulders, elbows
                (15, 16), (17, 18),          # wrists, pinkies
                (19, 20), (21, 22),          # index, thumbs
                (23, 24), (25, 26),          # hips, knees
                (27, 28), (29, 30),          # ankles, heels
                (31, 32),                     # foot index
            ]
            for l, r in lr_pairs:
                x[:, [l, r]] = x[:, [r, l]]
            x = x.reshape(WINDOW_FRAMES, -1)
        
        # 4. Scale augmentation
        if np.random.random() < 0.3:
            scale = np.random.uniform(0.9, 1.1)
            x = x * scale
        
        return x


# Stratified train/val split (80/20)
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(all_windows, all_labels))

train_windows = all_windows[train_idx]
train_labels = all_labels[train_idx]
val_windows = all_windows[val_idx]
val_labels = all_labels[val_idx]

print(f"Train: {len(train_windows)} windows")
print(f"Val:   {len(val_windows)} windows")

# Class distribution per split
for name, labs in [("Train", train_labels), ("Val", val_labels)]:
    counts = Counter(labs.tolist())
    print(f"\n{name}:")
    for i, cls in enumerate(GESTURE_CLASSES):
        print(f"  {cls}: {counts.get(i, 0)}")

# Create datasets and loaders
BATCH_SIZE = 64

train_ds = GestureDataset(train_windows, train_labels, augment=True)
val_ds = GestureDataset(val_windows, val_labels, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Verify shapes
batch_x, batch_y = next(iter(train_loader))
print(f"\nBatch X shape: {batch_x.shape}")  # (64, 15, 132)
print(f"Batch Y shape: {batch_y.shape}")      # (64,)

## Cell 8: Gesture Transformer Model

In [ ]:
import torch.nn as nn


class PositionalEncoding(nn.Module):
    """Learnable positional encoding."""
    def __init__(self, d_model, max_len=64):
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class GestureTransformer(nn.Module):
    """Transformer encoder for gesture classification.
    
    Input:  (batch, 15, 132)
    Output: (batch, 5)
    """
    def __init__(
        self,
        input_dim=132,
        d_model=256,
        nhead=4,
        num_layers=4,
        dim_feedforward=512,
        dropout=0.1,
        num_classes=5,
        max_seq_len=64,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        x = self.input_proj(x)       # (B, T, d_model)
        x = self.pos_encoding(x)
        x = self.encoder(x)          # (B, T, d_model)
        x = self.norm(x)
        x = x.mean(dim=1)            # mean pool → (B, d_model)
        return self.classifier(x)     # (B, num_classes)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GestureTransformer().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"GestureTransformer: {total_params:,} parameters")
print(f"Device: {device}")

# Verify forward pass
dummy = torch.randn(2, 15, 132).to(device)
out = model(dummy)
print(f"Output shape: {out.shape}")  # (2, 5)

## Cell 9: Training Loop

AdamW optimizer, cosine schedule, 30 epochs.
Should take ~30-60 minutes on T4.

In [ ]:
from sklearn.metrics import f1_score, classification_report
import time

# Class weights for imbalanced data
train_counts = Counter(train_labels.tolist())
total_train = len(train_labels)
class_weights = torch.tensor(
    [total_train / (5 * train_counts.get(i, 1)) for i in range(5)],
    dtype=torch.float32
).to(device)
print("Class weights:")
for i, cls in enumerate(GESTURE_CLASSES):
    print(f"  {cls}: {class_weights[i]:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)

NUM_EPOCHS = 30
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

best_val_f1 = 0.0
patience = 0
PATIENCE = 10


def evaluate(model, loader, device):
    """Compute macro F1 on a dataloader."""
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            preds = logits.argmax(dim=-1).cpu()
            all_preds.extend(preds.tolist())
            all_true.extend(y.tolist())
    return f1_score(all_true, all_preds, average='macro', zero_division=0), all_true, all_preds


print(f"\n{'='*60}")
print(f"Training: {NUM_EPOCHS} epochs, {len(train_loader)} batches/epoch")
print(f"{'='*60}\n")

train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    epoch_start = time.time()
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    val_f1, _, _ = evaluate(model, val_loader, device)
    epoch_time = time.time() - epoch_start
    lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} ({epoch_time:.0f}s) | "
          f"Loss: {train_loss:.4f} | Val F1: {val_f1:.4f} | LR: {lr:.2e}", end="")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'val_f1': best_val_f1,
            'classes': GESTURE_CLASSES,
        }, f"{SAVE_DIR}/best_model.pt")
        print(f" << BEST")
    else:
        patience += 1
        print(f" ({patience}/{PATIENCE})")
        if patience >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

total_time = time.time() - train_start
print(f"\nTraining complete in {total_time/60:.1f} minutes")
print(f"Best Val F1: {best_val_f1:.4f} (target >= 0.65)")

## Cell 10: Final Evaluation & Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix

# Load best model
ckpt = torch.load(f"{SAVE_DIR}/best_model.pt", map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best model from epoch {ckpt['epoch']}")

# Evaluate on full val set
val_f1, all_true, all_preds = evaluate(model, val_loader, device)

print(f"\n{'='*60}")
print(f"FINAL VALIDATION RESULTS")
print(f"{'='*60}")
print(f"Macro F1: {val_f1:.4f}")
print(f"Target:   >= 0.65")
print(f"Result:   {'PASSED' if val_f1 >= 0.65 else 'BELOW TARGET'}")

print(f"\n{classification_report(all_true, all_preds, target_names=GESTURE_CLASSES, zero_division=0)}")

# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
print("Confusion Matrix:")
print(f"{'':15s}", end="")
for cls in GESTURE_CLASSES:
    print(f"{cls[:6]:>8s}", end="")
print()
for i, cls in enumerate(GESTURE_CLASSES):
    print(f"{cls:15s}", end="")
    for j in range(5):
        print(f"{cm[i][j]:8d}", end="")
    print()

if val_f1 < 0.65:
    print("\nSuggestions to improve:")
    print("  1. Increase NUM_EPOCHS to 50")
    print("  2. Try lr=1e-4 (lower learning rate)")
    print("  3. Increase augmentation probability")
    print("  4. Check if class distribution is too skewed")
    print("  5. Review heuristic labels — may need manual correction")

## Cell 11: Save Model to Google Drive

In [ ]:
import json

# Save training metadata
with open(f"{SAVE_DIR}/training_meta.json", "w") as f:
    json.dump({
        "val_f1": float(val_f1),
        "best_epoch": int(ckpt['epoch']),
        "classes": GESTURE_CLASSES,
        "num_classes": 5,
        "total_windows": int(len(all_windows)),
        "train_windows": int(len(train_windows)),
        "val_windows": int(len(val_windows)),
        "num_videos": len(all_video_poses),
        "window_frames": WINDOW_FRAMES,
        "hop_frames": HOP_FRAMES,
        "model_params": sum(p.numel() for p in model.parameters()),
    }, f, indent=2)

# Also save the pose cache to Drive for future reuse
POSE_DRIVE_DIR = "/content/drive/MyDrive/voice_pipeline_models/gesture_transformer/pose_cache"
os.makedirs(POSE_DRIVE_DIR, exist_ok=True)
!cp /content/pose_cache/*.npz "{POSE_DRIVE_DIR}/"

print(f"\nSaved to {SAVE_DIR}/")
!ls -lh {SAVE_DIR}/

print(f"\n{'='*60}")
print(f"DONE! Download best_model.pt from Google Drive to your Mac:")
print(f"  Place at: ~/Desktop/Claude-assistant/models/gesture_transformer/best_model.pt")
print(f"{'='*60}")

## Cell 12 (Optional): Visualize Gesture Predictions on a Video

Pick one video and see what gestures the model detects over time.

In [ ]:
import matplotlib.pyplot as plt

# Pick first video
test_video = list(all_video_poses.keys())[0]
test_poses = all_video_poses[test_video]
print(f"Visualizing: {test_video}")

# Create windows
test_wins, test_labs, test_confs = create_labeled_windows(test_poses, WINDOW_FRAMES, HOP_FRAMES)
test_wins_tensor = torch.tensor(np.array(test_wins), dtype=torch.float32).to(device)

# Model predictions
model.eval()
with torch.no_grad():
    logits = model(test_wins_tensor)
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = logits.argmax(dim=-1).cpu().numpy()

# Time axis (in seconds)
times = np.arange(len(preds)) * HOP_FRAMES / TARGET_FPS

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Top: gesture predictions over time
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#95a5a6']
for i, cls in enumerate(GESTURE_CLASSES):
    mask = preds == i
    if mask.any():
        ax1.scatter(times[mask], [i] * mask.sum(), c=colors[i], s=30, label=cls, alpha=0.7)

ax1.set_yticks(range(5))
ax1.set_yticklabels(GESTURE_CLASSES)
ax1.set_ylabel('Gesture Class')
ax1.set_title(f'Gesture Timeline: {test_video}')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Bottom: class probabilities stacked
for i, cls in enumerate(GESTURE_CLASSES):
    ax2.plot(times, probs[:, i], label=cls, color=colors[i], alpha=0.8)

ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('Probability')
ax2.set_title('Class Probabilities')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/gesture_timeline.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved timeline plot to {SAVE_DIR}/gesture_timeline.png")